<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# LASSO Regression

LASSO continues the penalized-ERM idea from ridge regression, but changes the penalty. Ridge uses the squared $\ell_2$ norm; LASSO uses the $\ell_1$ norm
$$
\|w\|_1=\sum_{j=1}^D |w_j|.
$$
LASSO stands for **least absolute shrinkage and selection operator**. Assuming the predictors have been standardized and the response has been centered, define
$$
\boxed{
J_\lambda(w)
=\frac{1}{2N}\|y-Xw\|_2^2+\lambda\|w\|_1
}
$$
and
$$
\widehat w_\lambda=\arg\min_w J_\lambda(w).
$$

*Convention:* The factor $1/(2N)$ makes the later formulas cleaner and matches `sklearn`'s LASSO parameter `alpha`; changing this constant only rescales $\lambda$. An intercept is ordinarily fitted separately and left unpenalized.

At $\lambda=0$ we recover the least-squares problem. Larger $\lambda$ produces stronger shrinkage, and sufficiently large values drive all penalized coefficients to zero. Unlike ridge, LASSO can make individual coefficients **exactly zero**, so it performs shrinkage and variable selection together.

In [ ]:
#| code-fold: true
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Polygon
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

The two penalties have visibly different constraint regions. Their shape will later give a geometric explanation for the exact zeros.

In [ ]:
#| code-fold: true
theta = np.linspace(0, 2*np.pi, 500)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].fill(np.cos(theta), np.sin(theta), alpha=0.15)
axes[0].plot(np.cos(theta), np.sin(theta))
axes[0].set_title(r"Ridge: $\ell_2$ unit ball")

l1_boundary = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]])
axes[1].fill(l1_boundary[:, 0], l1_boundary[:, 1], alpha=0.15)
axes[1].plot(l1_boundary[:, 0], l1_boundary[:, 1])
axes[1].set_title(r"LASSO: $\ell_1$ unit ball")

for ax in axes:
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
    ax.set(xlabel=r"$w_1$", ylabel=r"$w_2$",
           xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), aspect="equal")
fig.tight_layout()
plt.show()

## What Changes Relative to Ridge?

The ridge penalty $\|w\|_2^2$ is differentiable, so its objective leads to normal equations and a matrix formula. The LASSO penalty contains $|w_j|$, which has a corner at $w_j=0$. The LASSO objective is convex but not differentiable everywhere, so there is generally no comparable closed-form matrix solution.

The corner is more than a computational inconvenience: it is exactly where zero can satisfy the LASSO optimality condition over a whole range of data values. To see this, we first need a derivative that works at a corner.

## Solving LASSO with Coordinate Descent

### Subgradients

For a convex function $h$, a number $q$ is a **subgradient** at $u$ if
$$
h(v)\geq h(u)+q(v-u)
\qquad\text{for every }v.
$$
Thus the line with slope $q$ through $(u,h(u))$ lies below the graph of $h$. The set of all such slopes is the **subdifferential**, written $\partial h(u)$.

In [ ]:
#| code-fold: true
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

We will compare two convex functions:

1. Absolute value,
   $$
   f(x)=|x|.
   $$
2. A piecewise-linear **dead-zone function**,
   $$
   f(x)=\max\{|x|-1,0\}.
   $$
   This function is flat on $[-1,1]$ and has corners at both endpoints.

In [ ]:
#| code-fold: true
def function_and_subdifferential(kind, x, x0):
    # Return f(x), f(x0), and the endpoint slopes of its subdifferential.
    if kind == "Absolute value":
        values = np.abs(x)
        value_at_x0 = abs(x0)
        if np.isclose(x0, 0.0):
            slope_low, slope_high = -1.0, 1.0
        elif x0 < 0:
            slope_low = slope_high = -1.0
        else:
            slope_low = slope_high = 1.0
        formula = r"$f(x)=|x|$"

    elif kind == "Dead zone":
        values = np.maximum(np.abs(x)-1.0, 0.0)
        value_at_x0 = max(abs(x0)-1.0, 0.0)
        if np.isclose(x0, -1.0):
            slope_low, slope_high = -1.0, 0.0
        elif np.isclose(x0, 1.0):
            slope_low, slope_high = 0.0, 1.0
        elif x0 < -1.0:
            slope_low = slope_high = -1.0
        elif x0 > 1.0:
            slope_low = slope_high = 1.0
        else:
            slope_low = slope_high = 0.0
        formula = r"$f(x)=\max\{|x|-1,0\}$"

    else:
        raise ValueError(f"Unknown function: {kind}")

    return values, value_at_x0, slope_low, slope_high, formula


def subdifferential_label(x0, slope_low, slope_high):
    if np.isclose(slope_low, slope_high):
        slope_set = rf"\{{{slope_low:g}\}}"
    else:
        slope_set = rf"[{slope_low:g},{slope_high:g}]"
    return rf"$\partial f({x0:g})={slope_set}$"


def draw_subgradients(ax, kind, x0, xlim=(-3, 3)):
    x = np.linspace(*xlim, 700)
    f, f0, slope_low, slope_high, formula = (
        function_and_subdifferential(kind, x, x0)
    )

    # The extreme slopes bound every supporting line in the subdifferential.
    line_low = f0+slope_low*(x-x0)
    line_high = f0+slope_high*(x-x0)
    swept_low = np.minimum(line_low, line_high)
    swept_high = np.maximum(line_low, line_high)

    if not np.isclose(slope_low, slope_high):
        ax.fill_between(
            x, swept_low, swept_high,
            color="tab:orange", alpha=0.18,
            label="region swept by all supporting lines",
        )
        slopes = np.linspace(slope_low, slope_high, 17)
    else:
        slopes = np.array([slope_low])

    colors = plt.cm.plasma(np.linspace(0.12, 0.88, len(slopes)))
    for slope, color in zip(slopes, colors):
        supporting_line = f0+slope*(x-x0)
        ax.plot(x, supporting_line, color=color, alpha=0.75, linewidth=1)

    ax.plot(x, f, color="black", linewidth=2.5, label=formula)
    ax.scatter([x0], [f0], color="crimson", s=55, zorder=5,
               label=r"selected point $(x_0,f(x_0))$")
    ax.axvline(x0, color="crimson", linestyle=":", linewidth=1)
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.set(
        xlim=xlim,
        ylim=(-3.2, 3.2),
        xlabel=r"$x$",
        ylabel=r"$y$",
        title=f"{kind} at " + rf"$x_0={x0:g}$",
    )
    ax.text(
        0.03, 0.95,
        subdifferential_label(x0, slope_low, slope_high),
        transform=ax.transAxes, ha="left", va="top",
        fontsize=11,
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )

    legend_handles = [
        Line2D([0], [0], color="black", linewidth=2.5,
               label="convex function"),
        Line2D([0], [0], color="tab:purple", linewidth=1.5,
               label="supporting line"),
        Line2D([0], [0], marker="o", linestyle="none",
               markerfacecolor="crimson", markeredgecolor="crimson",
               label="selected point"),
    ]
    if not np.isclose(slope_low, slope_high):
        legend_handles.insert(
            2,
            Patch(facecolor="tab:orange", alpha=0.18,
                  label="swept region"),
        )
    ax.legend(handles=legend_handles, loc="lower right", fontsize=8)

In [ ]:
examples = [
    ("Absolute value", 0.0),
    ("Absolute value", 1.5),
    ("Dead zone", 1.0),
    ("Dead zone", 0.0),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, (kind, x0) in zip(axes.flat, examples):
    draw_subgradients(ax, kind, x0)
fig.suptitle("Supporting lines at smooth points and corners", fontsize=15)
fig.tight_layout()
plt.show()

For absolute value,
$$
\partial |u|=
\begin{cases}
\{1\}, & u>0,\\
[-1,1], & u=0,\\
\{-1\}, & u<0.
\end{cases}
$$
Away from zero this agrees with the usual derivative. At zero every slope from $-1$ to $1$ supports the graph.

For a differentiable convex function, a point is a minimizer when its derivative is zero. The corresponding necessary-and-sufficient condition for a convex function with corners is
$$
0\in\partial h(u).
$$
We will apply this condition to one coefficient at a time.

### Coordinate Descent Setup

**Coordinate descent** updates one coefficient while holding all the others fixed:

  - Initialize $w$.
  - For $t=1,2,3,\ldots$:
    - For $j=1,\ldots,D$:
      1. Choose $\gamma_j\in\partial_j J_\lambda(w)$.
      2. Update
         $$
         w_j\leftarrow w_j-\eta_t \gamma_j.
         $$

  The updates occur in place. If $J_\lambda$ is differentiable, then
  $$
  \gamma_j=\frac{\partial J_\lambda(w)}{\partial w_j}.
  $$

Coordinate descent is guaranteed to converge under standard conditions when the objective is convex and separable across coordinates in its nonsmooth part, as it is for LASSO.

To do this for LASSO, suppose we are updating $w_j$, and write $X_{:j}$ for feature column $j$. Define the partial residual
$$
r_j
=y-\sum_{\ell\ne j}X_{:\ell}w_\ell.
$$
It removes the contribution of every feature except $j$. With the other coefficients fixed, the remaining one-dimensional problem is
$$
\min_{w_j}
\left\{
\frac{1}{2N}\|r_j-X_{:j}w_j\|_2^2+\lambda|w_j|
\right\}.
$$

Expand the squared-error term:
$$
\begin{aligned}
\frac{1}{2N}\|r_j-X_{:j}w_j\|_2^2
&=\frac{1}{2N}r_j^\top r_j
  -\frac1N w_jX_{:j}^\top r_j
  +\frac{1}{2N}w_j^2X_{:j}^\top X_{:j}.
\end{aligned}
$$
The first term does not depend on $w_j$, so it does not affect which value minimizes the expression. Define
$$
a_j=\frac1N X_{:j}^\top X_{:j},
\qquad
z_j=\frac1N X_{:j}^\top r_j.
$$
We assume $a_j>0$, meaning feature $j$ is not constant. The quantity $z_j$ measures the alignment between feature $j$ and the residual left by the other features (like a *covariance* term). $a_j$ is more like a *variance* term. The coordinate objective, up to an irrelevant constant, is
$$
g_j(w_j)=\frac12a_jw_j^2-z_jw_j+\lambda|w_j|.
$$
Its subdifferential is
$$
\partial g_j(w_j)
=a_jw_j-z_j+\lambda\,\partial|w_j|.
$$
We now find the value of $w_j$ for which this set contains zero (i.e. the coordinate update is the $w_j$ which makes $0\in\partial g_j(w_j)$). 

#### Case 1: $z_j>\lambda$

First test zero. At $w_j=0$,
$$
\partial g_j(0)=[-z_j-\lambda,-z_j+\lambda].
$$
If $z_j>\lambda$, the entire interval is negative, so it cannot contain zero.

A negative coefficient cannot work either. For $w_j<0$,
$$
\partial g_j(w_j)=\{a_jw_j-z_j-\lambda\},
$$
which is negative when $a_j>0$ and $z_j>\lambda$. The minimizer must therefore be positive. For $w_j>0$,
$$
\partial g_j(w_j)=\{a_jw_j-z_j+\lambda\}.
$$
Setting its only element equal to zero gives
$$
w_j=\frac{z_j-\lambda}{a_j}>0.
$$
The sign agrees with the case we assumed, so this is the minimizer.

#### Case 2: $z_j<-\lambda$

The symmetric argument forces the minimizer to be negative. In that region,
$$
0=a_jw_j-z_j-\lambda,
$$
and hence
$$
w_j=\frac{z_j+\lambda}{a_j}<0.
$$

#### Case 3: $|z_j|\leq\lambda$

At zero we again have
$$
\partial g_j(0)=[-z_j-\lambda,-z_j+\lambda].
$$
The condition $|z_j|\leq\lambda$ is exactly the condition that this interval contains zero. Because $g_j$ is convex, $w_j=0$ is therefore its minimizer.

### Soft-Thresholding

Combining the three cases gives
$$
w_j\leftarrow
\begin{cases}
\dfrac{z_j-\lambda}{a_j}, & z_j>\lambda,\\[6pt]
0, & |z_j|\leq\lambda,\\[6pt]
\dfrac{z_j+\lambda}{a_j}, & z_j<-\lambda.
\end{cases}
$$
Define the **soft-thresholding operator**
$$
S(z,\lambda)
=\operatorname{sign}(z)(|z|-\lambda)_+
=
\begin{cases}
z-\lambda, & z>\lambda,\\
0, & |z|\leq\lambda,\\
z+\lambda, & z<-\lambda.
\end{cases}
$$
Then the update is simply
$$
\boxed{
w_j\leftarrow\frac{S(z_j,\lambda)}{a_j}.
}
$$

The data supply the partial-residual alignment $z_j$, while the penalty removes $\lambda$ from its magnitude. If $|z_j|$ cannot overcome the threshold, zero satisfies the optimality condition and the feature drops out of the current fit.

For standardized predictors,
$$
a_j=\frac1N X_{:j}^\top X_{:j}=1,
$$
so the update reduces to $w_j\leftarrow S(z_j,\lambda)$.

### The Coordinate Descent Algorithm

1. Initialize $w$, often at $w=0$.
2. For $j=1,\ldots,D$, compute the current partial residual
   $$
   r_j=y-\sum_{\ell\ne j}X_{:\ell}w_\ell.
   $$
3. Compute
   $$
   z_j=\frac1N X_{:j}^\top r_j,
   \qquad
   a_j=\frac1N X_{:j}^\top X_{:j}.
   $$
4. Update
   $$
   w_j\leftarrow\frac{S(z_j,\lambda)}{a_j}.
   $$
5. Cycle through the coordinates until the coefficients stop changing appreciably.

Each update uses the latest values of the other coefficients. Since the LASSO objective is convex, repeated coordinate-wise minimization converges to a global solution under standard conditions.

A minimal implementation follows the derivation directly.

In [ ]:
#| code-fold: true
rng = np.random.default_rng(2026)
N_demo, D_demo = 100, 20
X_demo = rng.normal(size=(N_demo, D_demo))
w_true_demo = np.zeros(D_demo)
w_true_demo[[1, 4, 7]] = [2.5, -1.8, 1.2]
y_demo = X_demo @ w_true_demo + rng.normal(scale=0.7, size=N_demo)

X_demo = StandardScaler().fit_transform(X_demo)
y_demo = y_demo - y_demo.mean()

In [ ]:
def soft_threshold(z, lam):
    return np.sign(z)*max(abs(z) - lam, 0.0)


def lasso_coordinate_descent(X, y, lam, max_iter=1000, tol=1e-8):
    N, D = X.shape
    w = np.zeros(D)
    a = np.sum(X**2, axis=0)/N

    for iteration in range(max_iter):
        max_change = 0.0
        for j in range(D):
            old_w_j = w[j]
            # Add feature j back to the full residual to obtain r_j.
            r_j = y - X @ w + X[:, j]*old_w_j
            z_j = X[:, j] @ r_j/N
            w[j] = soft_threshold(z_j, lam)/a[j]
            max_change = max(max_change, abs(w[j] - old_w_j))

        if max_change < tol:
            break

    return w, iteration + 1

In [ ]:
lambda_demo = 0.10
w_cd, iterations = lasso_coordinate_descent(
    X_demo, y_demo, lambda_demo
)

sklearn_lasso = Lasso(
    alpha=lambda_demo,
    fit_intercept=False,
    max_iter=10000,
    tol=1e-10,
).fit(X_demo, y_demo)

print(f"Coordinate descent iterations: {iterations}")
print(f"Largest coefficient difference: "
      f"{np.max(np.abs(w_cd-sklearn_lasso.coef_)):.2e}")

comparison = pd.DataFrame({
    "true": w_true_demo,
    "our update": w_cd,
    "sklearn": sklearn_lasso.coef_,
})
comparison.loc[(comparison.abs() > 1e-3).any(axis=1)].round(3)

## Orthonormal Predictors

The special case of orthonormal feature columns makes the full solution transparent. Suppose
$$
\frac1N X^\top X=I_D,
\qquad
z=\frac1N X^\top y.
$$
Orthogonality removes all cross-coordinate terms, so $z_j$ is the OLS coefficient and
$$
\boxed{
\widehat w_{\lambda,j}=S(z_j,\lambda).
}
$$
Small OLS coefficients disappear; larger ones lose exactly $\lambda$ from their magnitude.

Under the same normalized-loss convention, ridge with penalty $(\lambda/2)\|w\|_2^2$ instead gives
$$
\widehat w_{\lambda,j}^{\mathrm{ridge}}
=\frac{z_j}{1+\lambda}.
$$
Ridge continuously rescales every coefficient, while LASSO first creates a zero region and then shrinks the remaining coefficients.

In [ ]:
#| code-fold: true
z_grid = np.linspace(-4, 4, 400)
lambda_plot = 1.0

plt.figure(figsize=(7, 5))
plt.plot(z_grid, z_grid, label="OLS")
plt.plot(z_grid,
         np.sign(z_grid)*np.maximum(np.abs(z_grid)-lambda_plot, 0),
         label=rf"LASSO, $\lambda={lambda_plot:g}$")
plt.plot(z_grid, z_grid/(1+lambda_plot), label="ridge")
plt.axhline(0, color="gray", linewidth=0.7)
plt.axvline(0, color="gray", linewidth=0.7)
plt.xlabel(r"OLS coefficient $z_j$")
plt.ylabel("penalized coefficient")
plt.title("Soft-thresholding versus ridge shrinkage")
plt.legend()
plt.show()

## Penalized and Constrained Forms

Let
$$
L(w)=\frac1{2N}\|y-Xw\|_2^2.
$$
LASSO can be written either as
$$
\min_w\{L(w)+\lambda\|w\|_1\}
$$
or as
$$
\min_w L(w)
\qquad\text{subject to}\qquad
\|w\|_1\leq t.
$$
Larger $\lambda$ and smaller $t$ both impose stronger regularization. As with ridge, the two formulations trace the same set of solutions, although their numerical tuning values are different.

One motivation is direct variable selection. The conventional notation $\|w\|_0$ counts the nonzero coefficients, but directly constraining this count creates a difficult nonconvex problem. The $\ell_1$ constraint is a tractable convex surrogate that encourages, without guaranteeing, a sparse solution.

### Geometry

In two dimensions, $\|w\|_1\leq t$ is a diamond. The constrained solution is the point in that diamond lying on the smallest attainable loss contour.

In [ ]:
#| code-fold: true
X_geom = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0],
])
y_geom = np.array([1.0, 1.0, 2.0, 2.5])
w_ols_geom = np.linalg.lstsq(X_geom, y_geom, rcond=None)[0]


def loss_geom(w1, w2):
    predictions = (X_geom[:, 0, None, None]*w1
                   + X_geom[:, 1, None, None]*w2)
    residuals = y_geom[:, None, None] - predictions
    return 0.5*np.mean(residuals**2, axis=0)


def lasso_solution_geom(lam):
    if lam == 0:
        return w_ols_geom
    return Lasso(
        alpha=lam, fit_intercept=False,
        max_iter=100000, tol=1e-10,
    ).fit(X_geom, y_geom).coef_


def constrained_solution_geom(t):
    if np.sum(np.abs(w_ols_geom)) <= t:
        return w_ols_geom, 0.0

    low, high = 0.0, 1.0
    while np.sum(np.abs(lasso_solution_geom(high))) > t:
        high *= 2
    for _ in range(50):
        middle = (low+high)/2
        if np.sum(np.abs(lasso_solution_geom(middle))) > t:
            low = middle
        else:
            high = middle
    lam = (low+high)/2
    return lasso_solution_geom(lam), lam


def l1_diamond(radius):
    return np.array([
        [radius, 0], [0, radius], [-radius, 0], [0, -radius]
    ])


def plot_lasso_geometry(mode="constrained", t=1.0, lam=0.2):
    grid = np.linspace(-2.5, 2.5, 350)
    W1, W2 = np.meshgrid(grid, grid)
    loss = loss_geom(W1, W2)

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    levels = np.linspace(loss.min(), np.percentile(loss, 90), 16)
    ax.contour(W1, W2, loss, levels=levels,
               colors="gray", linewidths=0.8)

    if mode == "constrained":
        w_star, corresponding_lambda = constrained_solution_geom(t)
        radius = t
        title = (rf"Constrained LASSO: $t={t:.2f}$ "
                 rf"($\lambda\approx{corresponding_lambda:.3g}$)")
    else:
        w_star = lasso_solution_geom(lam)
        radius = np.sum(np.abs(w_star))
        penalized = loss + lam*(np.abs(W1)+np.abs(W2))
        penalty_levels = np.linspace(
            penalized.min(), np.percentile(penalized, 80), 10
        )
        ax.contour(W1, W2, penalized, levels=penalty_levels,
                   colors="tab:orange", linestyles="dashed",
                   linewidths=0.7)
        title = rf"Penalized LASSO: $\lambda={lam:.2f}$"

    ax.add_patch(Polygon(
        l1_diamond(radius), closed=True, fill=False,
        color="tab:blue", linewidth=2, label=r"$\ell_1$ boundary",
    ))
    ax.scatter(*w_ols_geom, marker="*", s=120,
               color="black", label="OLS")
    ax.scatter(*w_star, s=55, color="crimson", label="LASSO")
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
    ax.set(xlabel=r"$w_1$", ylabel=r"$w_2$", title=title,
           xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), aspect="equal")
    ax.legend()
    plt.show()

In [ ]:
# A static example remains visible in non-interactive renders.
plot_lasso_geometry(mode="constrained", t=0.7)

Run the following cell in Jupyter to explore either formulation. The update button avoids recomputing the figure while a slider is moving.

In [ ]:
#| code-fold: true
try:
    from ipywidgets import interact_manual, Dropdown, FloatSlider

    interact_manual(
        plot_lasso_geometry,
        mode=Dropdown(
            options=["constrained", "penalized"],
            value="constrained",
            description="view",
        ),
        t=FloatSlider(
            value=0.7, min=0.05, max=4.0, step=0.05,
            description="t", continuous_update=False,
        ),
        lam=FloatSlider(
            value=0.2, min=0.0, max=5.0, step=0.05,
            description="lambda", continuous_update=False,
        ),
    )
except ImportError:
    print("Install ipywidgets to use the interactive geometry figure.")

When the smallest attainable contour touches a corner of the diamond, one coordinate is exactly zero. This is the geometric version of the subgradient calculation: at a coordinate-axis corner, the subdifferential contains a range of slopes, and that range can include zero. Ridge has a round boundary, so its tangency point generally does not lie exactly on an axis.

## Correlated Predictors

Suppose two feature columns are nearly identical:
$$
X_{:2}\approx X_{:1}.
$$
Many coefficient pairs then produce nearly the same fitted values. Ridge tends to distribute weight across the two predictors. LASSO may select one and set the other to zero because concentrated and distributed coefficient vectors can have the same $\ell_1$ norm.

This can make the selected variables unstable: a small change in the sample may change which member of a correlated group is retained even when predictive performance changes very little. The following Monte Carlo experiment generates five nearly interchangeable copies of one signal variable and records which copies LASSO selects.

In [ ]:
#| code-fold: true
rng = np.random.default_rng(2027)
repetitions, group_size = 200, 5
selected_group = np.zeros((repetitions, group_size), dtype=bool)

for b in range(repetitions):
    latent = rng.normal(size=60)
    signal_group = (
        latent[:, None]
        + 0.05*rng.normal(size=(60, group_size))
    )
    noise_features = rng.normal(size=(60, 10))
    X_mc = np.column_stack([signal_group, noise_features])
    y_mc = 3*latent + rng.normal(scale=1.5, size=60)

    X_mc = StandardScaler().fit_transform(X_mc)
    y_mc = y_mc-y_mc.mean()
    model = Lasso(
        alpha=0.2, fit_intercept=False,
        max_iter=20000, tol=1e-8,
    ).fit(X_mc, y_mc)
    selected_group[b] = np.abs(model.coef_[:group_size]) > 1e-8

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(selected_group[:40], cmap="Blues", aspect="auto",
               vmin=0, vmax=1)
axes[0].set(xlabel="correlated copy", ylabel="simulated sample",
            title="Selected copies in 40 samples",
            xticks=np.arange(group_size),
            xticklabels=np.arange(1, group_size+1))
axes[1].bar(np.arange(1, group_size+1), selected_group.mean(axis=0))
axes[1].set(xlabel="correlated copy", ylabel="selection frequency",
            title="No copy is consistently selected", ylim=(0, 1),
            xticks=np.arange(1, group_size+1))
fig.tight_layout()
plt.show()

Every copy carries essentially the same signal, yet different samples retain different subsets. LASSO selection is therefore evidence about a useful predictive representation, not automatic evidence that a selected variable is uniquely important or causal.

## Bias, Variance, and Sparsity

Increasing $\lambda$ generally raises bias, lowers variance, and reduces the number of selected variables. Moderate regularization can improve performance on new data; excessive regularization removes useful signal and underfits. The exact evaluation-error curve depends on the data and need not be perfectly U-shaped.

The following teaching simulation shows training error, held-out error, and selected-model size over an entire regularization path. Inspecting the held-out curve here illustrates typical behavior; it is not a valid procedure for choosing $\lambda$ in an actual analysis.

In [ ]:
#| code-fold: true
rng = np.random.default_rng(2028)
N_sim, D_sim = 100, 50
X_sim = rng.normal(size=(N_sim, D_sim))
w_true_sim = np.zeros(D_sim)
w_true_sim[:5] = [3, -2, 1.5, 0.5, -1]
y_sim = X_sim @ w_true_sim + rng.normal(scale=2.0, size=N_sim)

X_train_sim, X_test_sim, y_train_sim, y_test_sim = train_test_split(
    X_sim, y_sim, test_size=0.4, random_state=2028
)
lambda_sim = np.logspace(-4, 1, 140)
train_mse_sim, test_mse_sim, nonzero_sim = [], [], []

for lam in lambda_sim:
    model = make_pipeline(
        StandardScaler(),
        Lasso(alpha=lam, max_iter=100000),
    ).fit(X_train_sim, y_train_sim)
    train_mse_sim.append(mean_squared_error(
        y_train_sim, model.predict(X_train_sim)
    ))
    test_mse_sim.append(mean_squared_error(
        y_test_sim, model.predict(X_test_sim)
    ))
    nonzero_sim.append(np.count_nonzero(
        model.named_steps["lasso"].coef_
    ))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lambda_sim, train_mse_sim, label="training MSE")
axes[0].plot(lambda_sim, test_mse_sim, label="held-out MSE")
axes[0].set(xscale="log", xlabel=r"$\lambda$", ylabel="MSE",
            title="Fit and held-out performance")
axes[0].legend()
axes[1].plot(lambda_sim, nonzero_sim)
axes[1].set(xscale="log", xlabel=r"$\lambda$",
            ylabel="number of nonzero coefficients",
            title="Selected-model size")
fig.tight_layout()
plt.show()

## Practical Use

Standardize predictors so the penalty treats their units comparably, and place that scaling inside any evaluated pipeline. Software ordinarily leaves the intercept unpenalized. LASSO is most appealing when a sparse predictive representation is plausible, but selected variables can be unstable when predictors are strongly correlated.

Choose $\lambda$ with validation or cross-validation. When reporting final performance, keep the test data outside the entire selection procedure.

### Diabetes Data

The `sklearn` diabetes data contain $N=442$ patients and $D=10$ baseline measurements. The response is a quantitative measure of disease progression one year later. We reserve a test set, use five-fold CV on the remaining observations to choose $\lambda$, refit the selected pipeline on all model-building data, and evaluate it once on the test set.

In [ ]:
diabetes = load_diabetes()
X_diabetes = diabetes.data
y_diabetes = diabetes.target
feature_names = np.asarray(diabetes.feature_names)

X_build, X_test, y_build, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=2029
)
lambda_grid = np.logspace(-3, 2, 60)
lasso_pipeline = make_pipeline(
    StandardScaler(),
    Lasso(max_iter=100000, tol=1e-7),
)
folds = KFold(n_splits=5, shuffle=True, random_state=2030)
search = GridSearchCV(
    lasso_pipeline,
    param_grid={"lasso__alpha": lambda_grid},
    scoring="neg_mean_squared_error",
    cv=folds,
    refit=True,
).fit(X_build, y_build)

selected_lambda = search.best_params_["lasso__alpha"]
final_model = search.best_estimator_
test_mse = mean_squared_error(y_test, final_model.predict(X_test))
final_coefficients = final_model.named_steps["lasso"].coef_

print(f"N = {X_diabetes.shape[0]}, D = {X_diabetes.shape[1]}")
print(f"Selected lambda: {selected_lambda:.4g}")
print(f"Selected predictors: {np.count_nonzero(final_coefficients)}")
print(f"Final test MSE: {test_mse:.2f}")

In [ ]:
#| code-fold: true
cv_mean = -search.cv_results_["mean_test_score"]
cv_sd = search.cv_results_["std_test_score"]

scaler = final_model.named_steps["standardscaler"]
X_build_scaled = scaler.transform(X_build)
coefficient_path = []
for lam in lambda_grid:
    model = Lasso(
        alpha=lam, fit_intercept=True,
        max_iter=100000, tol=1e-7,
    ).fit(X_build_scaled, y_build)
    coefficient_path.append(model.coef_)
coefficient_path = np.asarray(coefficient_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(lambda_grid, cv_mean, label="mean CV MSE")
axes[0].fill_between(
    lambda_grid, cv_mean-cv_sd, cv_mean+cv_sd,
    alpha=0.2, label="± 1 fold SD",
)
axes[0].axvline(selected_lambda, color="black", linestyle="--",
                label=rf"selected $\lambda={selected_lambda:.3g}$")
axes[0].set(xscale="log", xlabel=r"$\lambda$",
            ylabel="cross-validated MSE", title=r"Selecting $\lambda$")
axes[0].legend()

for j, name in enumerate(feature_names):
    axes[1].plot(lambda_grid, coefficient_path[:, j], label=name)
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].axvline(selected_lambda, color="black", linestyle="--")
axes[1].set(xscale="log", xlabel=r"$\lambda$",
            ylabel=r"$\widehat w_{\lambda,j}$",
            title="Coefficient paths")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
coefficient_table = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": final_coefficients,
    })
    .assign(abs_coefficient=lambda d: d["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
    .drop(columns="abs_coefficient")
)
coefficient_table.loc[coefficient_table["coefficient"] != 0].round(3)

The CV scores select $\lambda$ as part of model building; the held-out MSE evaluates the refitted selected pipeline. As $\lambda$ increases along the path, coefficients reach zero at different values. For squared-error LASSO these paths are piecewise linear, although the displayed log scale bends the line segments visually.

## Extensions

**Elastic net** combines the LASSO and ridge penalties,
$$
\lambda\left[
\rho\|w\|_1+\frac{1-\rho}{2}\|w\|_2^2
\right],
\qquad 0\leq\rho\leq1,
$$
retaining sparsity while often behaving more stably with correlated predictors. The same penalties can be added to logistic cross-entropy instead of squared error, producing penalized logistic regression. The general recipe remains: choose a loss, add a penalty, and tune its strength using validation or cross-validation.

## Review Questions

See: @sec-lasso-questions.